In [1]:
import os
import random
from typing import Tuple, List, Dict

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm
import timm 
from sklearn.metrics import (
    accuracy_score, 
    roc_auc_score, 
    precision_recall_fscore_support, 
)

In [3]:
from utils.config import (
    MODEL_NAME, NUM_CLASSES, BATCH_SIZE, EVAL_BATCH_SIZE, NUM_WORKERS, DEVICE,
    LEARNING_RATE, WEIGHT_DECAY, NUM_EPOCHS, T_MAX_LR_SCHEDULER_EPOCHS, CHECKPOINT_PATH,
    TRAIN_DIR, TEST_DIR, VAL_DIR, CHECKPOINT_PATH_1
)

In [4]:
from utils.dataset import ChestXrayDataset, train_tf, val_tf, get_file_paths_and_labels

#### Set seeds for reproducibility

In [5]:
def set_seed(seed: int = 42) -> None:
    """Sets the seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        # For deterministic behavior
        torch.backends.cudnn.deterministic = True 
        torch.backends.cudnn.benchmark = False
    print(f"Seeds set to {seed}.")

#### -------------------- Model Definition --------------------

In [6]:
def get_model(model_name: str, num_classes: int, pretrained: bool = True) -> nn.Module:
    """Instantiates a Vision Transformer (ViT) model from timm."""
    
    # Use timm to create the model, using pretrained ImageNet weights
    model = timm.create_model(
        model_name, 
        pretrained=pretrained, 
        num_classes=num_classes
    )
    
    # Example for freezing: Freeze all layers except the classification head.
    # This is a common practice for transfer learning in early epochs.
    # for name, param in model.named_parameters():
    #      if 'head' not in name:
    #          param.requires_grad = False
             
    print(f"Model: {model_name} instantiated. Number of classes: {num_classes}")
    return model

#### -------------------- Training and Evaluation Functions --------------------
##### 1. Training

In [7]:
def train_epoch(
    model: nn.Module, 
    loader: DataLoader, 
    optimizer: torch.optim.Optimizer, 
    criterion: nn.Module, 
    device: str,
    scaler: torch.cuda.amp.GradScaler
) -> Tuple[float, float]:
    """Runs a single training epoch."""
    model.train()
    losses: List[float] = []
    all_preds: List[int] = []
    all_labels: List[int] = []
    
    # Use enumerate for tracking progress more clearly if needed, but tqdm is sufficient
    for images, labels in tqdm(loader, desc="Training"):
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        # Mixed Precision Training
        with torch.autocast(device_type=device, dtype=torch.float16):
            logits = model(images)
            loss = criterion(logits, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        # Metrics collection
        losses.append(loss.item())
        # Use .detach().cpu() only when necessary for non-gradient operations
        preds = torch.argmax(logits.detach().cpu(), dim=1).numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy()) # labels are already on device, move back for numpy
        
    acc = accuracy_score(all_labels, all_preds)
    return float(np.mean(losses)), float(acc)

##### 2. Evaluating

In [8]:
def eval_epoch(
    model: nn.Module, 
    loader: DataLoader, 
    criterion: nn.Module, 
    device: str
) -> Dict[str, float]:
    """Runs a single evaluation epoch and returns comprehensive metrics."""
    model.eval()
    losses: List[float] = []
    all_probs: List[float] = []
    all_preds: List[int] = []
    all_labels: List[int] = []
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating"):
            images = images.to(device)
            labels = labels.to(device)
            
            with torch.autocast(device_type=device, dtype=torch.float16):
                logits = model(images)
                loss = criterion(logits, labels)
                # Softmax to get probabilities for AUC, choosing the positive class (index 1)
                probs = torch.softmax(logits, dim=1)[:, 1] 
                
            losses.append(loss.item())
            
            # Metrics collection
            preds = torch.argmax(logits.cpu(), dim=1).numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            
    # Calculate comprehensive metrics
    avg_loss = np.mean(losses)
    acc = accuracy_score(all_labels, all_preds)
    
    try:
        # AUC requires probabilities for the positive class
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        # Happens if only one class is present in the batch/dataset (rare, but good to handle)
        auc = 0.0
        
    # precision, recall, f1 for binary classification
    prec, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='binary', zero_division=0
    )
    
    # Optional: Confusion Matrix
    # cm = confusion_matrix(all_labels, all_preds)
    
    return {
        'loss': avg_loss,
        'accuracy': acc,
        'auc': auc,
        'precision': prec,
        'recall': recall,
        'f1': f1,
    }

### -------------------- Main Training Loop --------------------

##### 1. Data Preparation

In [9]:
set_seed(42)
try:
    train_files, train_labels, train_weights_np = get_file_paths_and_labels(TRAIN_DIR)
    # val_files, val_labels, _ = get_file_paths_and_labels(VAL_DIR)
    val_files, val_labels, _ = get_file_paths_and_labels(TEST_DIR)
    
except FileNotFoundError as e:
    print(f"Error: Data directory not found. Please update BASE_DIR in config.py.")
    print(f"Missing directory: {e}")

Seeds set to 42.
Loaded 5216 samples from C:\Users\HP\Desktop\SLIIT\Y4 SEM 1\DL\Ass\Assignment\DL-project\ViT\dataset\chest_xray\train. Class counts: {'NORMAL': 1341, 'PNEUMONIA': 3875}
Calculated class weights: [1.9448173  0.67303226] (Index 0: NORMAL, Index 1: PNEUMONIA)
Loaded 624 samples from C:\Users\HP\Desktop\SLIIT\Y4 SEM 1\DL\Ass\Assignment\DL-project\ViT\dataset\chest_xray\test. Class counts: {'NORMAL': 234, 'PNEUMONIA': 390}
Calculated class weights: [1.33333333 0.8       ] (Index 0: NORMAL, Index 1: PNEUMONIA)


##### 2. Datasets and DataLoaders

In [10]:
train_ds = ChestXrayDataset(train_files, train_labels, transform=train_tf)
val_ds   = ChestXrayDataset(val_files, val_labels, transform=val_tf)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True
)
val_loader   = DataLoader(
    val_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)
print("DataLoaders initialized.")

DataLoaders initialized.


##### 3. Model, Loss, Optimizer, and Scheduler

In [11]:
model = get_model(MODEL_NAME, NUM_CLASSES).to(DEVICE)

# Move class weights to the device and convert to torch.float
class_weights = torch.tensor(train_weights_np, dtype=torch.float32).to(DEVICE) 
criterion = nn.CrossEntropyLoss(weight=class_weights)
    
# Filter only parameters that are set to require gradients (e.g., if we froze layers)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), 
    lr=LEARNING_RATE, 
    weight_decay=WEIGHT_DECAY
)

# NOTE: The original T_max=20 assumes T_max is per-epoch changed to per-step for better L.R. control.
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=T_MAX_LR_SCHEDULER_EPOCHS * len(train_loader) # T_max is number of steps
) 

Model: vit_base_patch16_224 instantiated. Number of classes: 2


##### 4. Training Loop

In [ ]:
scaler = torch.cuda.amp.GradScaler() # Mixed precision scaler

best_val_auc = 0.0
for epoch in range(1, NUM_EPOCHS + 1):
    # TRAIN
    train_loss, train_acc = train_epoch(
            model, train_loader, optimizer, criterion, DEVICE, scaler
    )
        
    # VALIDATE
    val_metrics = eval_epoch(model, val_loader, criterion, DEVICE)
    val_loss, val_acc, val_auc, val_prec, val_rec, val_f1 = (
        val_metrics['loss'], val_metrics['accuracy'], val_metrics['auc'], 
        val_metrics['precision'], val_metrics['recall'], val_metrics['f1']
    )
        
    # SCHEDULER STEP
    scheduler.step()
        
    # LOGGING
    print(
        f"Epoch {epoch}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}, Val AUC: {val_auc:.4f}, "
        f"Val Recall: {val_rec:.4f}, Val F1: {val_f1:.4f}"
    )
        
    # SAVE BEST MODEL
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save({
            'epoch': epoch,
            'model_state': model.state_dict(), 
            'optimizer_state': optimizer.state_dict(),
            'best_val_auc': best_val_auc
        }, CHECKPOINT_PATH_1)   
        # Or CHECKPOINT_PATH as prev 
        print(f"--- Model saved! New best AUC: {best_val_auc:.4f} ---")

C:\Users\HP\AppData\Local\Temp\ipykernel_12340\4072805567.py:1: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() # Mixed precision scaler
Validating: 100%|██████████| 20/20 [00:10<00:00,  1.85it/s]


Epoch 1/30 | Train Loss: 0.3669, Train Acc: 0.8585 | Val Loss: 0.8553, Val Acc: 0.8269, Val AUC: 0.9351, Val Recall: 0.9744, Val F1: 0.8756
--- Model saved! New best AUC: 0.9351 ---


Validating: 100%|██████████| 20/20 [00:11<00:00,  1.69it/s]


Epoch 2/30 | Train Loss: 0.2118, Train Acc: 0.9172 | Val Loss: 1.8824, Val Acc: 0.7484, Val AUC: 0.9422, Val Recall: 0.9974, Val F1: 0.8321
--- Model saved! New best AUC: 0.9422 ---


Validating: 100%|██████████| 20/20 [00:11<00:00,  1.69it/s]


Epoch 3/30 | Train Loss: 0.1576, Train Acc: 0.9388 | Val Loss: 0.5604, Val Acc: 0.8846, Val AUC: 0.9455, Val Recall: 0.9641, Val F1: 0.9126
--- Model saved! New best AUC: 0.9455 ---


Training:  51%|█████     | 167/326 [01:13<00:51,  3.10it/s]